# 03: DNA Language Model Perplexity Benchmark on *An. gambiae* Coding Sequences

This notebook runs the same perplexity benchmark as `02_perplexity_benchmark.ipynb` but restricts the input sequences to **spliced protein-coding sequences (CDS)** from chromosome 3L, rather than random windows.

**Motivating question.** In notebook 02, both NT-v2 and HyenaDNA showed weak transfer to *An. gambiae* chromosomal DNA. NT-v2's mean loss (8.560) was slightly above the random baseline (8.320), suggesting the model was confidently wrong more often than it was correct. One possible explanation is that DNA foundation models trained on diverse genomes learn protein-coding grammar (codon usage, stop codons, splice junctions) more strongly than non-coding sequence patterns. Chromosome 3L is mostly non-coding (~85%). If the coding-grammar hypothesis is correct, restricting the input to coding sequences only should produce substantially better predictions.

**Two possible outcomes:**

- **Coding regions produce much lower loss** than random 3L windows: supports the hypothesis that foundation models learned coding-region patterns but did not learn non-coding mosquito content.
- **Coding regions produce similar loss** to random 3L windows: transfer failure is species-specific rather than sequence-type-specific.

Both outcomes are publishable. The comparison at the end of this notebook is designed to distinguish them.

**Data sources:**
- Reference genome: `D:\Mathieson_data\AgamP4.fa` (extracted from Ensembl Metazoa release-58)
- Gene annotations: `D:\Mathieson_data\Anopheles_gambiae.AgamP4.58.gtf` (same source, gene model file)

**Kernel:** Python 3.11 (Mathieson-LM)

## Setup

Standard imports plus the addition of the GTF file path. HuggingFace cache is redirected to the project's `models/` directory so downloaded model weights are self-contained.

In [ ]:
# Standard imports
import sys, os, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Project path resolution ---
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

# --- External data paths (kept on D: to save C: drive space) ---
REFERENCE_PATH = Path(r'D:\Mathieson_data\AgamP4.fa')
GTF_PATH       = Path(r'D:\Mathieson_data\Anopheles_gambiae.AgamP4.58.gtf')

# --- HuggingFace cache configuration ---
os.environ['HF_HOME'] = str(MODELS_DIR)

# --- Scientific Python and ML imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# --- Verify both data files exist ---
for name, path in [('Reference genome', REFERENCE_PATH), ('GTF annotation', GTF_PATH)]:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'{name:<20}: {path}  ({size_mb:.1f} MB)')
    else:
        print(f'WARNING: {name} not found at {path}')

## Parsing the GTF to find CDS entries on chromosome 3L

GTF is a tab-separated text format where each row describes a genomic feature (gene, transcript, exon, CDS, etc.). The columns are:

1. `seqname`: chromosome or scaffold name
2. `source`: annotation provider (VectorBase, Ensembl, etc.)
3. `feature`: feature type (`gene`, `transcript`, `CDS`, `exon`, `start_codon`, `stop_codon`, etc.)
4. `start`: 1-based inclusive start coordinate
5. `end`: 1-based inclusive end coordinate
6. `score`: usually `.`
7. `strand`: `+` or `-` indicating which strand the feature is on
8. `frame`: reading frame offset (0, 1, or 2) for CDS features
9. `attribute`: semicolon-separated key-value pairs (`gene_id`, `transcript_id`, etc.)

This cell reads the file with pandas, filters to CDS features on chromosome 3L, and extracts the `transcript_id` from the attributes column so we can group CDS entries by their parent transcript.

In [ ]:
import re

# --- Read the GTF file directly with pandas ---
# GTF header lines start with '#'; pandas can skip them with comment='#'
# The nine standard columns have canonical names in the bioinformatics community
GTF_COLUMNS = ['seqname', 'source', 'feature', 'start', 'end',
               'score', 'strand', 'frame', 'attribute']

print('Reading GTF file...')
gtf = pd.read_csv(
    GTF_PATH,
    sep='\t',
    comment='#',
    header=None,
    names=GTF_COLUMNS,
    dtype={'seqname': str, 'start': int, 'end': int, 'strand': str, 'feature': str}
)
print(f'  Total rows in GTF: {len(gtf):,}')

# --- Filter to CDS entries on chromosome 3L only ---
# We keep only CDS features (not gene/transcript/exon/UTR entries) because
# CDS gives the exact coordinates of the protein-coding portions
cds_3l = gtf[(gtf['feature'] == 'CDS') & (gtf['seqname'] == '3L')].copy()
print(f'  CDS entries on 3L: {len(cds_3l):,}')

# --- Extract transcript_id from the attribute column ---
# Attribute format looks like: 'gene_id "AGAP008283"; transcript_id "AGAP008283-RA"; ...'
# A simple regex pulls out the transcript_id value
def extract_transcript_id(attribute_str):
    match = re.search(r'transcript_id "([^"]+)"', attribute_str)
    return match.group(1) if match else None

cds_3l['transcript_id'] = cds_3l['attribute'].apply(extract_transcript_id)
print(f'  Unique transcripts on 3L with CDS: {cds_3l["transcript_id"].nunique():,}')

## Building spliced CDS sequences per transcript

In eukaryotes, most genes have multiple exons separated by introns. The GTF file lists each CDS exon on its own row. To get the actual biological protein-coding sequence for a gene, its CDS exons must be:

1. **Sorted by genomic coordinate.**
2. **Concatenated in the direction of transcription** (which reverses for genes on the negative strand).
3. **Reverse-complemented if on the negative strand**, so the sequence matches how the protein is actually read.

This cell processes each transcript on 3L, joins its CDS exons into a single spliced sequence, and filters to transcripts with total coding length at least 500 bp so we can sample 500 bp windows for perplexity evaluation.

In [ ]:
import pyfaidx

# --- Open the reference genome for random access ---
fasta = pyfaidx.Fasta(str(REFERENCE_PATH))
chrom_3L = fasta['3L']
print(f'Chromosome 3L length: {len(chrom_3L):,} bp')

def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence.
    A <-> T, C <-> G, N stays as N. Sequence is reversed after complementing."""
    complement = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
    return seq.translate(complement)[::-1]


def build_spliced_cds(cds_rows):
    """
    Given the CDS DataFrame rows for a single transcript, return the spliced
    coding sequence as an uppercase string on the transcribed strand.

    Steps:
      1. Sort exons by start coordinate.
      2. Extract each exon's sequence from the reference (coordinates are 1-based inclusive).
      3. Concatenate.
      4. If the transcript is on the negative strand, reverse-complement the whole spliced sequence.
    """
    strand = cds_rows['strand'].iloc[0]
    exons_sorted = cds_rows.sort_values('start')

    # pyfaidx uses 0-based half-open slicing; GTF is 1-based inclusive.
    # So GTF start=100 end=105 (bases 100-105 inclusive) becomes pyfaidx [99:105].
    exon_strs = []
    for _, row in exons_sorted.iterrows():
        exon_strs.append(str(chrom_3L[row['start'] - 1 : row['end']]).upper())

    spliced = ''.join(exon_strs)

    # Negative-strand transcripts are read in reverse: reverse-complement the full spliced sequence
    if strand == '-':
        spliced = reverse_complement(spliced)

    return spliced


# --- Build spliced CDS for every transcript on 3L ---
# groupby preserves per-transcript grouping so we can process each independently
print('Building spliced CDS sequences per transcript...')
spliced_cds = {}
for transcript_id, group in cds_3l.groupby('transcript_id'):
    spliced_cds[transcript_id] = build_spliced_cds(group)

# --- Report basic stats ---
cds_lengths = pd.Series({t: len(s) for t, s in spliced_cds.items()})
print(f'\nTotal transcripts on 3L: {len(spliced_cds):,}')
print(f'CDS length distribution (bp):')
print(f'  min:    {cds_lengths.min():>8,}')
print(f'  median: {int(cds_lengths.median()):>8,}')
print(f'  mean:   {int(cds_lengths.mean()):>8,}')
print(f'  max:    {cds_lengths.max():>8,}')
print(f'\nTranscripts with CDS >= 500 bp: {(cds_lengths >= 500).sum():,}')

## Sampling 500 bp windows from coding sequences

To make the perplexity numbers directly comparable to notebook 02 (which used 500 bp random genomic windows), we sample 500 bp windows from within the spliced CDS sequences. Each sampled sequence:

- Comes from a distinct transcript (no double-sampling from the same gene)
- Is exactly 500 bp of pure coding sequence (no introns, no UTRs)
- Is drawn from a random offset within its parent transcript

The random seed matches notebook 02 for a controlled comparison.

In [ ]:
def sample_cds_windows(spliced_cds_dict, n=50, length=500, seed=42):
    """
    Sample n random 500 bp windows, each from a distinct transcript.

    Parameters
    ----------
    spliced_cds_dict : dict
        Mapping transcript_id -> spliced CDS string.
    n : int
        Number of sequences to sample.
    length : int
        Window length in base pairs. Only transcripts with CDS >= this length are eligible.
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    list of str
        n uppercase coding sequences of the requested length.
    """
    # Filter to eligible transcripts (spliced CDS long enough to sample a window)
    eligible = [t for t, s in spliced_cds_dict.items() if len(s) >= length]
    if len(eligible) < n:
        raise ValueError(f'Only {len(eligible)} eligible transcripts, need {n}')

    rng = np.random.default_rng(seed)

    # Pick n transcripts without replacement (each sequence from a distinct gene)
    chosen_transcripts = rng.choice(eligible, size=n, replace=False)

    sequences = []
    for transcript_id in chosen_transcripts:
        cds = spliced_cds_dict[transcript_id]
        # Sample a random start within the transcript that leaves room for the window
        max_start = len(cds) - length
        start = int(rng.integers(0, max_start + 1))
        sequences.append(cds[start:start + length])

    return sequences

# --- Sample 50 coding sequences of length 500 bp ---
sequences = sample_cds_windows(spliced_cds, n=50, length=500)

print(f'Sampled {len(sequences)} coding sequences of length {len(sequences[0])} bp')
print(f'First sequence (first 80 bp): {sequences[0][:80]}...')

# Empirical GC content across the batch
actual_gc = np.mean([(s.count('G') + s.count('C')) / len(s) for s in sequences])
print(f'Empirical GC content: {actual_gc:.2%}  (expected ~50%+ for coding regions in An. gambiae, higher than the ~44% genome-wide average)')

## Loss computation helpers

Same two functions as notebook 02, unchanged. `compute_mlm_loss` handles masked language models (NT-v2), `compute_causal_loss` handles autoregressive models (HyenaDNA). Reproduced here to keep the notebook self-contained.

In [ ]:
def compute_mlm_loss(model, tokenizer, sequences, mask_prob=0.15, device='cuda', max_length=None):
    """Compute average masked-LM loss across a list of DNA sequences."""
    model.eval()
    model = model.to(device)
    losses = []

    with torch.no_grad():
        for seq in sequences:
            # Tokenize with truncation to the model's max context
            enc = tokenizer(seq, return_tensors='pt', truncation=True,
                            max_length=max_length or tokenizer.model_max_length)
            input_ids = enc['input_ids'].to(device)

            # Identify non-special-token positions eligible for masking
            special_mask = torch.tensor(
                tokenizer.get_special_tokens_mask(input_ids[0].tolist(), already_has_special_tokens=True),
                dtype=torch.bool, device=device
            )
            mask_candidates = ~special_mask

            # Randomly choose 15% of candidate positions to mask
            rand = torch.rand(input_ids.shape[1], device=device)
            mask_positions = mask_candidates & (rand < mask_prob)

            if mask_positions.sum() == 0:
                continue

            # Replace masked positions with the tokenizer's [MASK] token ID
            masked_input_ids = input_ids.clone()
            masked_input_ids[0, mask_positions] = tokenizer.mask_token_id

            # Loss labels: -100 is ignored by cross-entropy; only masked positions contribute
            loss_labels = torch.full_like(input_ids, -100)
            loss_labels[0, mask_positions] = input_ids[0, mask_positions]

            outputs = model(input_ids=masked_input_ids, labels=loss_labels)
            losses.append(outputs.loss.item())

    return float(np.mean(losses))


def compute_causal_loss(model, tokenizer, sequences, device='cuda', max_length=None):
    """Compute average next-token (causal) loss across a list of DNA sequences."""
    model.eval()
    model = model.to(device)
    losses = []

    with torch.no_grad():
        for seq in sequences:
            enc = tokenizer(seq, return_tensors='pt', truncation=True,
                            max_length=max_length or tokenizer.model_max_length)
            input_ids = enc['input_ids'].to(device)

            # For causal LM, labels are the input_ids themselves (model shifts internally)
            outputs = model(input_ids=input_ids, labels=input_ids)
            losses.append(outputs.loss.item())

    return float(np.mean(losses))

print('Helper functions defined.')

## Nucleotide Transformer v2 evaluation on coding sequences

Model weights are already cached from notebook 02, so this load is fast. The perplexity is computed on the 50 coding-sequence windows sampled above.

In [ ]:
NT_MODEL_ID = 'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species'

print(f'Loading {NT_MODEL_ID}...')
nt_tokenizer = AutoTokenizer.from_pretrained(NT_MODEL_ID, trust_remote_code=True)
nt_model = AutoModelForMaskedLM.from_pretrained(NT_MODEL_ID, trust_remote_code=True)
print(f'Loaded. Vocab size: {nt_tokenizer.vocab_size:,}')

print('\nComputing MLM loss on 50 coding sequences...')
nt_loss_coding = compute_mlm_loss(nt_model, nt_tokenizer, sequences, device=DEVICE)

print(f'\nNT-v2 mean MLM loss (coding):       {nt_loss_coding:.4f}')
print(f'NT-v2 perplexity (coding):          {np.exp(nt_loss_coding):.2f}')
print(f'Random-baseline reference:          {np.log(nt_tokenizer.vocab_size):.4f}  (worst case)')

# Free VRAM for the next model
del nt_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## DNABERT-2 evaluation on coding sequences

Same BPE-tokenized MLM as in notebook 02. Uses the direct-class-import loading path (via `get_class_from_dynamic_module`) to bypass the AutoModel registry conflict that would otherwise prevent DNABERT-2 from loading with its checkpoint weights properly mapped. See notebook 02's DNABERT-2 cell for the full explanation.

In [ ]:
DNABERT2_MODEL_ID = 'zhihan1996/DNABERT-2-117M'

# --- Custom loading path (see notebook 02 for the full rationale) ---
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

print(f'Loading {DNABERT2_MODEL_ID}...')

dnabert2_tokenizer = AutoTokenizer.from_pretrained(DNABERT2_MODEL_ID, trust_remote_code=True)
dnabert2_config    = AutoConfig.from_pretrained(DNABERT2_MODEL_ID, trust_remote_code=True)

# Import DNABERT-2's own BertForMaskedLM class directly from cached remote code
BertForMaskedLM_dnabert2 = get_class_from_dynamic_module(
    'bert_layers.BertForMaskedLM',
    DNABERT2_MODEL_ID,
)

# Load with custom class + custom config so all 117M parameters map cleanly
dnabert2_model = BertForMaskedLM_dnabert2.from_pretrained(DNABERT2_MODEL_ID, config=dnabert2_config)
print(f'Loaded. Vocab size: {dnabert2_tokenizer.vocab_size:,}')
print(f'Total parameters: {sum(p.numel() for p in dnabert2_model.parameters()):,}')

# --- Compute mean MLM loss on the coding sequences ---
print('\nComputing MLM loss on 50 coding sequences...')
dnabert2_loss_coding = compute_mlm_loss(dnabert2_model, dnabert2_tokenizer, sequences, device=DEVICE)

print(f'\nDNABERT-2 mean MLM loss (coding):   {dnabert2_loss_coding:.4f}')
print(f'DNABERT-2 perplexity (coding):      {np.exp(dnabert2_loss_coding):.2f}')
print(f'Random-baseline reference:          {np.log(dnabert2_tokenizer.vocab_size):.4f}  (worst case)')

# Free VRAM for the next model
del dnabert2_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## HyenaDNA evaluation on coding sequences

Same causal-LM setup as notebook 02, run on the coding sequences.

In [ ]:
# --- Notebook 02 numbers, hard-coded for direct comparison ---
# Update these after rerunning notebook 02 with the DNABERT-2 fix.
# DNABERT2_LOSS_RANDOM should be filled in with the value printed by notebook 02's DNABERT-2 cell.
NT_LOSS_RANDOM        = 8.560
DNABERT2_LOSS_RANDOM  = None    # TODO: fill in after rerunning notebook 02
HYENA_LOSS_RANDOM     = 1.400

# --- Build comparison DataFrame ---
# Only include rows where the random-loss counterpart is available
rows = [
    {'model': 'NT-v2 (6-mer, MLM)',           'sequence_type': 'random 3L',   'loss': NT_LOSS_RANDOM,       'vocab_size': nt_tokenizer.vocab_size,       'baseline': np.log(nt_tokenizer.vocab_size)},
    {'model': 'NT-v2 (6-mer, MLM)',           'sequence_type': 'coding 3L',   'loss': nt_loss_coding,       'vocab_size': nt_tokenizer.vocab_size,       'baseline': np.log(nt_tokenizer.vocab_size)},
    {'model': 'DNABERT-2 (BPE, MLM)',         'sequence_type': 'random 3L',   'loss': DNABERT2_LOSS_RANDOM, 'vocab_size': dnabert2_tokenizer.vocab_size, 'baseline': np.log(dnabert2_tokenizer.vocab_size)},
    {'model': 'DNABERT-2 (BPE, MLM)',         'sequence_type': 'coding 3L',   'loss': dnabert2_loss_coding, 'vocab_size': dnabert2_tokenizer.vocab_size, 'baseline': np.log(dnabert2_tokenizer.vocab_size)},
    {'model': 'HyenaDNA (single-nt, causal)', 'sequence_type': 'random 3L',   'loss': HYENA_LOSS_RANDOM,    'vocab_size': hyena_tokenizer.vocab_size,    'baseline': np.log(hyena_tokenizer.vocab_size)},
    {'model': 'HyenaDNA (single-nt, causal)', 'sequence_type': 'coding 3L',   'loss': hyena_loss_coding,    'vocab_size': hyena_tokenizer.vocab_size,    'baseline': np.log(hyena_tokenizer.vocab_size)},
]

# Drop any row with None loss (e.g., DNABERT-2 random before notebook 02 has been rerun)
rows = [r for r in rows if r['loss'] is not None]

comparison = pd.DataFrame(rows)
comparison['perplexity'] = np.exp(comparison['loss'])
print(comparison.to_string(index=False))

# --- Grouped bar chart: random vs coding, per model ---
fig, ax = plt.subplots(figsize=(11, 6))

models_unique = comparison['model'].unique()
x = np.arange(len(models_unique))
bar_width = 0.35

# For each model, pull the random and coding losses (None if not available)
random_losses = []
coding_losses = []
baselines     = []
for m in models_unique:
    sub = comparison[comparison['model'] == m]
    r = sub[sub['sequence_type'] == 'random 3L']['loss']
    c = sub[sub['sequence_type'] == 'coding 3L']['loss']
    random_losses.append(r.iloc[0] if len(r) else np.nan)
    coding_losses.append(c.iloc[0] if len(c) else np.nan)
    baselines.append(sub['baseline'].iloc[0])

b1 = ax.bar(x - bar_width/2, random_losses, bar_width, label='random 3L windows',   color='tab:gray')
b2 = ax.bar(x + bar_width/2, coding_losses, bar_width, label='coding 3L sequences', color='tab:green')

# Overlay random-baseline lines (dashed) per model
for i, base in enumerate(baselines):
    ax.plot([i - 0.4, i + 0.4], [base, base], color='tab:red', linestyle='--', linewidth=1)
    ax.text(i + 0.4, base, f' random baseline: {base:.2f}', color='tab:red', va='center', fontsize=8)

# Numeric labels on top of each bar (skip NaN bars)
for bar, val in zip(b1, random_losses):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}', ha='center', va='bottom')
for bar, val in zip(b2, coding_losses):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}', ha='center', va='bottom')

ax.set_xticks(x)
ax.set_xticklabels(models_unique)
ax.set_ylabel('Mean loss (cross-entropy per predicted token)')
ax.set_title('DNA-LM perplexity: random 3L windows vs. coding 3L sequences')
ax.legend(loc='upper left')
plt.tight_layout()

out_path = OUTPUTS_DIR / 'perplexity_random_vs_coding.png'
plt.savefig(out_path, dpi=120)
print(f'\nSaved figure to {out_path}')
plt.show()

# --- Notify if any random-loss counterparts are missing ---
missing = [m for m, l in zip(models_unique, random_losses) if np.isnan(l)]
if missing:
    print(f'\nMissing random-loss values (need to be filled in above): {missing}')

## Comparison to notebook 02 (random 3L windows)

The direct comparison between random-window and coding-region perplexity is the scientific point of this notebook. The random-window numbers below are taken from notebook 02's most recent run. Update the constants if that run changes.

**How to read the delta:**

- Large drop (coding much lower than random): supports the hypothesis that DNA foundation models learned protein-coding grammar and use it here.
- Small or no drop: transfer failure is species-specific rather than sequence-type-specific.
- Increase (coding higher than random): would be very surprising and would warrant re-checking the coding-sequence extraction.

In [ ]:
# --- Notebook 02 numbers, hard-coded for direct comparison ---
# Update these if you rerun notebook 02 and get different values.
NT_LOSS_RANDOM     = 8.560
HYENA_LOSS_RANDOM  = 1.400

# --- Build comparison DataFrame ---
comparison = pd.DataFrame([
    {'model': 'NT-v2 (6-mer, MLM)',           'sequence_type': 'random 3L',   'loss': NT_LOSS_RANDOM,     'perplexity': np.exp(NT_LOSS_RANDOM),     'vocab_size': nt_tokenizer.vocab_size,     'baseline': np.log(nt_tokenizer.vocab_size)},
    {'model': 'NT-v2 (6-mer, MLM)',           'sequence_type': 'coding 3L',   'loss': nt_loss_coding,     'perplexity': np.exp(nt_loss_coding),     'vocab_size': nt_tokenizer.vocab_size,     'baseline': np.log(nt_tokenizer.vocab_size)},
    {'model': 'HyenaDNA (single-nt, causal)', 'sequence_type': 'random 3L',   'loss': HYENA_LOSS_RANDOM,  'perplexity': np.exp(HYENA_LOSS_RANDOM),  'vocab_size': hyena_tokenizer.vocab_size,  'baseline': np.log(hyena_tokenizer.vocab_size)},
    {'model': 'HyenaDNA (single-nt, causal)', 'sequence_type': 'coding 3L',   'loss': hyena_loss_coding,  'perplexity': np.exp(hyena_loss_coding),  'vocab_size': hyena_tokenizer.vocab_size,  'baseline': np.log(hyena_tokenizer.vocab_size)},
])
print(comparison.to_string(index=False))

# --- Grouped bar chart: random vs coding, per model ---
fig, ax = plt.subplots(figsize=(10, 6))

models_unique = comparison['model'].unique()
x = np.arange(len(models_unique))
bar_width = 0.35

random_losses = [comparison[(comparison['model'] == m) & (comparison['sequence_type'] == 'random 3L')]['loss'].iloc[0] for m in models_unique]
coding_losses = [comparison[(comparison['model'] == m) & (comparison['sequence_type'] == 'coding 3L')]['loss'].iloc[0] for m in models_unique]
baselines     = [comparison[comparison['model'] == m]['baseline'].iloc[0] for m in models_unique]

b1 = ax.bar(x - bar_width/2, random_losses, bar_width, label='random 3L windows', color='tab:gray')
b2 = ax.bar(x + bar_width/2, coding_losses, bar_width, label='coding 3L sequences', color='tab:green')

# Overlay random-baseline lines (dashed) per model to show where 'no better than uniform' sits
for i, base in enumerate(baselines):
    ax.plot([i - 0.4, i + 0.4], [base, base], color='tab:red', linestyle='--', linewidth=1)
    ax.text(i + 0.4, base, f' random baseline: {base:.2f}', color='tab:red', va='center', fontsize=8)

# Numeric labels on top of each bar
for bar, val in zip(b1, random_losses):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}', ha='center', va='bottom')
for bar, val in zip(b2, coding_losses):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}', ha='center', va='bottom')

ax.set_xticks(x)
ax.set_xticklabels(models_unique)
ax.set_ylabel('Mean loss (cross-entropy per predicted token)')
ax.set_title('DNA-LM perplexity: random 3L windows vs. coding 3L sequences')
ax.legend(loc='upper left')
plt.tight_layout()

out_path = OUTPUTS_DIR / 'perplexity_random_vs_coding.png'
plt.savefig(out_path, dpi=120)
print(f'\nSaved figure to {out_path}')
plt.show()

## Interpretation

The comparison depends on the direction and magnitude of the loss delta between random and coding for each model.

**If NT-v2 coding loss dropped substantially (e.g., from 8.56 to below 5.0):**
- Consistent with the hypothesis that NT-v2 learned protein-coding grammar during pretraining.
- Non-coding mosquito DNA is out-of-distribution for NT-v2 because its training corpus was likely coding-biased.
- Suggests domain-adapted pretraining on non-coding regions would substantially help.
- Publishable framing: 'DNA foundation models transfer to coding but not non-coding regions in *An. gambiae*, motivating region-aware pretraining strategies.'

**If NT-v2 coding loss stayed near random (say 7.5-8.5):**
- Coding vs non-coding is not the axis of failure.
- Transfer failure is species-specific: NT-v2 does not know *An. gambiae* patterns at all, regardless of coding status.
- Suggests species-specific fine-tuning would be needed.
- Publishable framing: 'DNA foundation models trained on 850 genomes show weak transfer to *An. gambiae* across both coding and non-coding regions, motivating species-specific pretraining or fine-tuning.'

**If HyenaDNA behaves similarly to NT-v2 across the coding/random split:**
- Suggests the finding generalizes across model families (bidirectional MLM and autoregressive causal).
- Strengthens the case that the issue is about training-data coverage, not model architecture.

**If HyenaDNA and NT-v2 diverge sharply on coding (e.g., one drops, the other does not):**
- The finding is architecture-dependent, which is more nuanced but still informative.

**Next step regardless of outcome:** the numbers here plus the existing notebook 02 numbers give enough concrete material to draft a paragraph for a follow-up email to Sara Mathieson. The framing writes itself once the direction of the delta is known.